# 01 — Trino quickstart

Trino is de SQL-engine van het platform. Vier catalogs:

| Catalog | Wat | Wie ziet wat |
|---|---|---|
| `bronze` | Onbewerkte brondata, incl. PII | JIT-toegang via OPA |
| `silver` | Geconformeerd, gepseudonimiseerd | meest gebruikers |
| `gold` | CGM-conforme business-products | dashboards & analyses |
| `sensitive` | Bijzondere persoonsgegevens (AVG art. 9) | 4-eyes |

Onder de motorkap: Delta-tabellen op MinIO, met Hive Metastore als
catalog-backend. OPA krijgt voor iedere query je rol en past row-filters,
kolom-maskers en doelbinding toe.

In [ ]:
from uwv_lab import trino

trino.query('SHOW CATALOGS')

In [ ]:
# Welke schemas zitten er in silver?
trino.query('SHOW SCHEMAS FROM silver')

In [ ]:
# Eerste 20 rijen uit een silver-tabel. OPA bepaalt welke kolommen je ziet.
trino.query('''
SELECT *
FROM silver.uwv.persona_created
LIMIT 20
''')

In [ ]:
# Direct als SQLAlchemy-engine — handig voor pandas read_sql.
import pandas as pd
engine = trino.sqlalchemy_engine()
pd.read_sql('SELECT count(*) FROM silver.uwv.persona_created', engine)

## Tip: identiteit-debug

Wil je weten welke user Trino jou ziet?

```python
trino.query('SELECT current_user')
```

Dat moet je Keycloak `preferred_username` zijn. Als je een `permission
denied` krijgt, hoort daar in de logs van OPA een denied-record te staan.